#### A typical RAG system searches uploaded documents to find a relevant snippet. Agentic RAG, instead of always looking up the information, the agent stops to consider whether it really needs to search or if it can answer on its own.

##### Here will be build a privacy-friendly Agentic RAG pipeline using Python, LangChain, and a lightweight Google model.
- LangChain to manage the process
- ChromaDB for storing vectors 
- Google’s Flan-T5 as the local language model

In [1]:
import os
from huggingface_hub import InferenceClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

/root/opt/ra-i-g/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_1039186/1623642887.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Setuo the model and the inference client

client = InferenceClient(token=os.getenv("HF_TOKEN"))
MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

def ask_llm(prompt, max_tokens=200):
    result = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        model=MODEL,
        max_tokens=max_tokens
    )
    return result.choices[0].message.content.strip()

In [ ]:
# Load PDFs from a folder
def load_docs(folder_path):
    docs = []
    # loading pdf files one page at a time
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(folder_path, file))
            docs.extend(loader.load())
    return docs

docs = load_docs("../data/raw/pdf")
print("PDF Pages Loaded:", len(docs))

# Chunking / spilitting the documents into smaller pieces

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # LLMs can only read a certain amount of text at once - the context window
    chunk_overlap=80    # Instead of just cutting the text, we let the chunks overlap a bit. Sentences aren’t split in half at the edge of a chunk, so the meaning (or semantic context) is preserved across breaks.
)
chunks = text_splitter.split_documents(docs)
print("Chunks Created:", len(chunks))

PDF Pages Loaded: 3
Chunks Created: 13


In [ ]:
# Embeddings and Vector Store - convert text into numbers (vectors) and sore it in chroma vector database for retrieval
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") # ll-MiniLM-L6-v2 - a small, fast model 

# Save texts into Chroma vector DB
texts = [c.page_content for c in chunks]
db = Chroma(
    collection_name="rag_store",
    embedding_function=embedding_model
)
db.add_texts(texts)

# Retriever
retriever = db.as_retriever(search_kwargs={"k": 3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13039.55it/s]


In [5]:
# Agent Controller

# Instead of sending everything to the database, this controller analyzes the user’s intent:
# - Does the user want data from the file? Action: Search
# - Is the user just chatting or asking for general knowledge? Action: Direct

def agent_controller(query):
    q = query.lower()
    if any(word in q for word in ["pdf", "document", "data", "summarize",
                                   "information", "find", "what does",
                                   "according to", "from the"]):
        return "search"
    return "direct"

In [6]:
# RAG : Question - Answer

def rag_answer(query):
    action = agent_controller(query)

    if action == "search":
        print(f"  Agent decided to SEARCH documents for: '{query}'")
        results = retriever.invoke(query)
        context = "\n".join([r.page_content for r in results])
        prompt = f"""Use this context to answer the question.
                    If the answer is not in the context, say "I don't have that information."

                    Context:
                    {context}

                    Question:
                    {query}

                    Answer:"""
    else:
        print(f"  Agent decided to answer DIRECTLY: '{query}'")
        prompt = query

    return ask_llm(prompt)

In [8]:
# Testing
print(rag_answer("Give me a 5-point summary from the PDF"))
print("-" * 50)
print(rag_answer("What is an Ideal Colour for painting ships? Explain in 50 words."))

  Agent decided to SEARCH documents for: 'Give me a 5-point summary from the PDF'
I don't have that information.

The provided context is a page about the Moon in our Skies, and it appears to be a collection of quotes and images related to lunar eclipses. There is no PDF mentioned in the context, and therefore, I cannot provide a 5-point summary from a non-existent PDF. If you meant to ask about the context provided, I can try to summarize it for you.
--------------------------------------------------
  Agent decided to answer DIRECTLY: 'What is an Ideal Colour for painting ships? Explain in 50 words.'
An ideal color for painting ships is a matte, non-reflective finish, often referred to as "flat" or "matt" paint. This color reduces glare and improves visibility, making it easier to spot ships at sea. A popular choice is a dark blue or grey, which also helps to conceal dirt and grime.
